In [ ]:
!pip install -q crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

In [ ]:
from crewai import LLM

llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.2
)

In [ ]:
%%writefile transcript.txt
Meeting: Final Year Project Sync - Smart Attendance App
Date: 17 October 2026
Attendees: Kavya, Rahul, Sneha, Arjun

Kavya: Okay let's start. We have the review with our guide on 30th October, so we need a working demo by then.
Rahul: The face recognition model is mostly working, but accuracy drops when lighting is bad. I think around 82 percent right now.
Sneha: Can we just add a fallback where students scan a QR code if face recognition fails?
Kavya: Yes, I like that. Let's go with QR code as the fallback. Everyone agree?
Arjun: Agreed.
Rahul: Fine with me.
Kavya: Good, that's decided then. Rahul, can you push accuracy above 90 percent by 24th October?
Rahul: I'll try, but 24th is tight. Can I get till 26th?
Kavya: Okay, 26th October then.
Arjun: For the backend, should we use Firebase or build our own Spring Boot server?
Sneha: Firebase is faster to set up.
Arjun: But Spring Boot looks better for placements.
Kavya: Let's go with Firebase for now since time is short. We can migrate later if needed.
Arjun: Okay, Firebase. I'll set up the database and authentication by 22nd October.
Sneha: I'll design the app screens in Figma by 20th October.
Rahul: Maybe we should also add a dashboard for teachers?
Kavya: Good idea, but let's think about it after the review. Not now.
Kavya: Also someone needs to write the project report draft.
Arjun: We'll figure that out next week.
Kavya: Okay. Next meeting on 21st October, same time. Thanks everyone.

Writing transcript.txt


In [ ]:
from crewai_tools import FileReadTool, FileWriterTool

read_tool = FileReadTool()
file_writer = FileWriterTool()

print(read_tool.run(file_path="transcript.txt"))

Meeting: Final Year Project Sync - Smart Attendance App
Date: 17 October 2026
Attendees: Kavya, Rahul, Sneha, Arjun

Kavya: Okay let's start. We have the review with our guide on 30th October, so we need a working demo by then.
Rahul: The face recognition model is mostly working, but accuracy drops when lighting is bad. I think around 82 percent right now.
Sneha: Can we just add a fallback where students scan a QR code if face recognition fails?
Kavya: Yes, I like that. Let's go with QR code as the fallback. Everyone agree?
Arjun: Agreed.
Rahul: Fine with me.
Kavya: Good, that's decided then. Rahul, can you push accuracy above 90 percent by 24th October?
Rahul: I'll try, but 24th is tight. Can I get till 26th?
Kavya: Okay, 26th October then.
Arjun: For the backend, should we use Firebase or build our own Spring Boot server?
Sneha: Firebase is faster to set up.
Arjun: But Spring Boot looks better for placements.
Kavya: Let's go with Firebase for now since time is short. We can migrate l

In [ ]:
from crewai import Agent

read_tool = FileReadTool(file_path="transcript.txt")

summarizer = Agent(
    role="Meeting Summarizer",
    goal="Write a clear 5-line summary of the meeting",
    backstory="You attend many meetings and can capture the main points "
              "in a few lines without missing anything important.",
    tools=[read_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

action_finder = Agent(
    role="Action Item Tracker",
    goal="Find every action item with its owner and FINAL due date",
    backstory="You are a careful project coordinator. If a deadline changes "
              "during a meeting, you always use the final one. If a task has "
              "no owner, you clearly mark it as 'Unassigned'.",
    tools=[read_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

decision_logger = Agent(
    role="Decision Logger",
    goal="Record only the decisions the team actually agreed on",
    backstory="You are strict. Ideas that were only suggested, postponed or "
              "rejected are NOT decisions. You list them separately as "
              "'Parked ideas'.",
    tools=[read_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

formatter = Agent(
    role="Notes Formatter",
    goal="Combine the summary, action items and decisions into clean, "
         "well-organised meeting notes in Markdown",
    backstory="You make documents easy to scan, with clear headings and "
              "tables. You never add information that was not given to you.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

In [ ]:
from crewai import Task

summary_task = Task(
    description="Read the meeting transcript using your tool and write a "
                "summary of the meeting in exactly 5 lines.",
    expected_output="A 5-line summary of the meeting.",
    agent=summarizer,
    async_execution=True,
)

actions_task = Task(
    description="Read the meeting transcript using your tool and list every "
                "action item. For each one give the task, the owner and the "
                "FINAL due date. If a deadline was changed, use the final one. "
                "If no one took responsibility, write 'Unassigned'.",
    expected_output="A list of action items, each with task, owner and due date.",
    agent=action_finder,
    async_execution=True,
)

decisions_task = Task(
    description="Read the meeting transcript using your tool and list only the "
                "decisions the team clearly agreed on. Do NOT include task "
                "assignments like 'X will do Y by date'; those are action items, "
                "not decisions. Then, separately, list any ideas that were "
                "suggested but postponed or not agreed, under 'Parked ideas'.",
    expected_output="A list of decisions, followed by a list of parked ideas.",
    agent=decision_logger,
    async_execution=True,
)

format_task = Task(
    description="Combine the summary, action items and decisions into clean "
                "meeting notes. Use these sections: Summary, Action Items "
                "(as a table with Task, Owner, Due Date), Decisions, Parked Ideas. "
                "If the same item appears in more than one section, keep it only "
                "in the most suitable section. Any task that still needs to be "
                "done must stay in Action Items, even if it has no owner; show "
                "the owner as 'Unassigned'.",
    expected_output="Well-formatted meeting notes in Markdown.",
    agent=formatter,
    context=[summary_task, actions_task, decisions_task],
    output_file="notes.md",
)

In [ ]:
from crewai import Crew, Process

notes_crew = Crew(
    agents = [summarizer, action_finder, decision_logger, formatter],
    tasks = [summary_task, actions_task, decisions_task, format_task],
    process = Process.sequential,
    max_rpm = 8,
    verbose = True,
)
result = await notes_crew.kickoff_async()

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 8d52767d-0bcb-4980-b021-e4a722afb589                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Read the meeting transcript using your tool and list every action     │
│  item. For each one give th

In [ ]:
import os
from IPython.display import Markdown, display

print("File exists:", os.path.exists("notes.md"))
with open("notes.md") as f:
    display(Markdown(f.read()))

File exists: True


# Meeting Notes: Smart Attendance App Sync

## Summary
The team held a sync meeting to prepare a working demo of the Smart Attendance App for their project guide review on October 30th. A follow-up meeting is scheduled for October 21st.

## Action Items

| Task | Owner | Due Date |
| :--- | :--- | :--- |
| Push face recognition accuracy above 90 percent | Rahul | 26th October |
| Set up the database and authentication using Firebase | Arjun | 22nd October |
| Design the app screens in Figma | Sneha | 20th October |
| Write the project report draft | Unassigned | Unassigned |

## Decisions
- Improve the face recognition model's accuracy, with a QR code introduced as a fallback option when face recognition fails.
- Set up the database and authentication using Firebase, prioritizing speed over building a custom Spring Boot server.

## Parked Ideas
- Add a dashboard for teachers (postponed until after the initial review).